# Statistics for ML Quality: Is It Actually Better, or Just Noise?

A teammate trained two candidate classifiers for a breast-cancer diagnosis model and wants to
ship whichever one "scored higher" on the held-out test set. As the QA reviewer, a single
accuracy number isn't a sign-off — you need to know whether the gap between the two models is
real or just sampling noise, and whether the data you evaluated on even looks like the data the
model will see later.

**What you should produce:**

1. `bootstrap_ci(y_true, y_pred, metric_fn, n_boot, ci, rng)` — a reusable bootstrap confidence
   interval for any sklearn-style metric function, returning `(point_estimate, lower, upper)`.
2. Two trained classifiers (`LogisticRegression` and `DecisionTreeClassifier`) evaluated with
   bootstrap CIs on accuracy and F1.
3. `paired_bootstrap_diff(...)` — a **paired** bootstrap test for the difference between the two
   models' accuracy, used to decide whether the gap is statistically significant (does the CI of
   the difference exclude zero?).
4. `ks_drift_check(reference, current, alpha)` — a Kolmogorov-Smirnov two-sample drift check,
   applied to a realistic train/production-style split of the test data, with a short written
   conclusion on whether drift was detected and what you'd do about it.

Dataset: `sklearn.datasets.load_breast_cancer` — a real diagnostic dataset (569 patients, 30
tumor measurements, malignant/benign label). It's a good fit here because it's an actual
quality-sign-off scenario (a diagnostic classifier), not a toy problem.


## Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score
from scipy import stats

RNG = np.random.default_rng(42)


## Load the data

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target  # 0 = malignant, 1 = benign

print(X.shape)
X.head()


## Part 1 — Bootstrap confidence interval for a metric

Any single accuracy or F1 number computed on a finite test set is itself a random variable — a
different test sample would give a slightly different number. Implement a generic bootstrap CI:
resample `(y_true, y_pred)` pairs together (so each resample keeps correct label/prediction
pairing), recompute the metric on each resample, and take percentiles of the resulting
distribution.


In [ ]:
def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=1000, ci=0.95, rng=None):
    """TODO: implement.

    Returns (point_estimate, lower_bound, upper_bound).
    """
    # TODO: compute the point estimate of metric_fn on the full (y_true, y_pred)
    # TODO: draw n_boot resamples of the *indices*, with replacement, same size as the data
    # TODO: for each resample, compute metric_fn on the resampled y_true/y_pred pair
    # TODO: take the [ (1-ci)/2, 1-(1-ci)/2 ] percentiles of the bootstrap distribution
    raise NotImplementedError


## Part 2 — Train two candidate models and compare them

Split the data into train/test, fit a `LogisticRegression` and a `DecisionTreeClassifier` on the
training set, and get predictions on the test set for both.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# TODO: fit a LogisticRegression(max_iter=5000) on (X_train, y_train)
# TODO: fit a DecisionTreeClassifier(random_state=42) on (X_train, y_train)
# TODO: predict on X_test with both models

model_a = ...  # LogisticRegression
model_b = ...  # DecisionTreeClassifier
preds_a = ...
preds_b = ...

# TODO: use bootstrap_ci to report accuracy and F1 (with 95% CI) for each model


## Part 3 — Is the difference significant, or just noise?

It's tempting to just compare the two point estimates and ship whichever is higher. Instead,
bootstrap the **difference** in accuracy directly, resampling the *same* indices for both
models on each draw (this keeps the comparison paired — both models are scored on the exact same
resampled patients each time, which is more powerful than treating the two accuracies as
independent samples). If the resulting confidence interval for `acc(model_a) - acc(model_b)`
excludes zero, the difference is statistically significant at that confidence level.


In [ ]:
def paired_bootstrap_diff(y_true, preds_a, preds_b, metric_fn, n_boot=2000, ci=0.95, rng=None):
    """TODO: implement.

    Returns (point_estimate_of_difference, lower_bound, upper_bound) for
    metric_fn(y_true, preds_a) - metric_fn(y_true, preds_b).
    """
    # TODO: compute the point estimate of the difference on the full data
    # TODO: for each of n_boot draws, resample one set of indices and apply it to
    #       y_true, preds_a, AND preds_b (this is what makes it "paired")
    # TODO: compute the metric difference on each resample, collect into an array
    # TODO: return the point estimate and the percentile interval of the differences
    raise NotImplementedError

# TODO: call paired_bootstrap_diff on accuracy, print the CI, and decide:
# is the difference statistically significant (does the CI exclude 0)?


## Part 4 — Drift check: does the test set look like production will?

A model can pass every offline metric and still fail in production if the population it sees
later doesn't look like the population it was evaluated on. Simulate a realistic version of this
check: sort the test set by one feature (`worst area`) and split it into two halves — a
"reference" half and a "current" half. Because tumor size correlates with other measurements,
sorting by `worst area` will shift the distribution of other features between the two halves,
the same way a real covariate shift (e.g. a change in patient demographics or imaging equipment
over time) would. Use a two-sample Kolmogorov-Smirnov test on `mean smoothness` between the two
halves to check whether that shift is statistically detectable.


In [ ]:
def ks_drift_check(reference, current, alpha=0.05):
    """TODO: implement.

    Returns a dict: {"statistic": ..., "p_value": ..., "drift_detected": bool}.
    """
    # TODO: run scipy.stats.ks_2samp(reference, current)
    # TODO: flag drift_detected = True if p_value < alpha
    raise NotImplementedError

# TODO:
# 1. sort a copy of X_test by "worst area"
# 2. split it into two halves (reference = first half, current = second half)
# 3. run ks_drift_check on the "mean smoothness" column of each half
# 4. print the result and write one sentence: was drift detected, and what would
#    you do next in a real monitoring pipeline if it was?


---

## Solution

*(scroll down when you're ready — try it yourself first)*


<details>
<summary>Click to reveal solution</summary>

**Part 1 — bootstrap CI**

```python
def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=1000, ci=0.95, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)

    point = metric_fn(y_true, y_pred)
    boot_stats = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_stats[i] = metric_fn(y_true[idx], y_pred[idx])

    alpha = 1 - ci
    lower, upper = np.percentile(boot_stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return point, lower, upper
```

**Part 2 — train and evaluate both models**

```python
model_a = LogisticRegression(max_iter=5000).fit(X_train, y_train)
model_b = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
preds_a = model_a.predict(X_test)
preds_b = model_b.predict(X_test)

for name, preds in [("LogisticRegression", preds_a), ("DecisionTree", preds_b)]:
    acc, acc_lo, acc_hi = bootstrap_ci(y_test, preds, accuracy_score, rng=RNG)
    f1, f1_lo, f1_hi = bootstrap_ci(y_test, preds, f1_score, rng=RNG)
    print(f"{name:18s} accuracy={acc:.3f} [{acc_lo:.3f}, {acc_hi:.3f}]   "
          f"f1={f1:.3f} [{f1_lo:.3f}, {f1_hi:.3f}]")
```

Accuracy alone can be misleading on a dataset like this (roughly 63/37 class balance) — a model
that always predicts "benign" would still score well over 60% accuracy, which is why F1 (or
recall on the malignant class) is reported alongside it rather than instead of it.

**Part 3 — paired bootstrap for the difference**

```python
def paired_bootstrap_diff(y_true, preds_a, preds_b, metric_fn, n_boot=2000, ci=0.95, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    y_true = np.asarray(y_true)
    preds_a = np.asarray(preds_a)
    preds_b = np.asarray(preds_b)
    n = len(y_true)

    point = metric_fn(y_true, preds_a) - metric_fn(y_true, preds_b)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        diffs[i] = metric_fn(y_true[idx], preds_a[idx]) - metric_fn(y_true[idx], preds_b[idx])

    alpha = 1 - ci
    lower, upper = np.percentile(diffs, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return point, lower, upper

diff, lo, hi = paired_bootstrap_diff(y_test, preds_a, preds_b, accuracy_score, rng=RNG)
print(f"accuracy(LogReg) - accuracy(Tree) = {diff:.3f}   95% CI [{lo:.3f}, {hi:.3f}]")
print("statistically significant:", lo > 0 or hi < 0)
```

Resampling the *same* indices for both models on every draw is what makes this a paired test —
it isolates the variance that comes from "which patients happened to be in the test set" while
keeping the direct model-vs-model comparison on matched samples, similar in spirit to a paired
t-test. On this split the two models are often close enough that the CI straddles zero, meaning
you cannot confidently say one is better — exactly the kind of result that should block a
"ship the higher number" decision.

**Part 4 — KS drift check**

```python
def ks_drift_check(reference, current, alpha=0.05):
    statistic, p_value = stats.ks_2samp(reference, current)
    return {"statistic": statistic, "p_value": p_value, "drift_detected": p_value < alpha}

sorted_by_area = X_test.sort_values("worst area")
half = len(sorted_by_area) // 2
reference = sorted_by_area.iloc[:half]["mean smoothness"]
current = sorted_by_area.iloc[half:]["mean smoothness"]

result = ks_drift_check(reference, current)
print(result)
```

Sorting by `worst area` before splitting manufactures a realistic covariate-shift scenario:
tumor size correlates with several other measurements, so the two halves end up with
systematically different distributions of `mean smoothness` even though every row came from the
same original dataset. The KS test typically reports a small p-value here (well under 0.05),
correctly flagging `drift_detected = True`. In a real monitoring pipeline this is the same check
you'd run per-feature between a training reference window and a rolling production window —
usually with a multiple-comparisons correction (e.g. Bonferroni) across features before alerting,
since testing dozens of features at `alpha=0.05` each will produce false alarms by chance alone.

</details>
